In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

import database

# Main

In [ ]:
API_KEY = os.environ['KEY']
SERIES_IDS = ['UNRATE', 'PAYEMS', 'ICSA', 'INDPRO', 'JTSJOL']

## Get data for db table `FREDSeries`
- SeriesID
- Title 
- DefaultFrequency
- Units
- ObservationStart
- FRED_LastUpdateLocalTime
- FRED_LastUpdateUTC (must be converted from CD/CDT to UTC)
- CreationDate `TEXT NOT NULL DEFAULT (datetime('now'))`

<b style="color:#fa0240;">Handle missing entries fetched from FRED</b>

In [ ]:
fred_time_stamp = "2026-08-07 08:31:41-05"
fred_time = datetime.strptime(fred_time_stamp + "00", "%Y-%m-%d %H:%M:%S%z")
utc_time = fred_time.astimezone(timezone.utc)

print(fred_time)
print(utc_time)

In [ ]:
fred_series_params = {
    "api_key": API_KEY,
    "file_type": "json"
}

def requester_fred_series(
        url="https://api.stlouisfed.org/fred/series",
        base_params=fred_series_params,
        SeriesIDs=SERIES_IDS
    ):
    results = []

    with requests.Session() as session:
        for series_id in SERIES_IDS:
            params = {
                **base_params,
                "series_id": series_id
            }

            r = session.get(url=url, params=params, timeout=30)
            r.raise_for_status()

            data = r.json()

            fred_local_time = datetime.strptime(data['seriess'][0]['last_updated'] + "00", "%Y-%m-%d %H:%M:%S%z")
            fred_utc_time = fred_local_time.astimezone(timezone.utc)

            fred_series_data = {
                "series_id": data['seriess'][0]['id'],
                "title": data['seriess'][0]['title'],
                "frequency": data['seriess'][0]['frequency'],
                "units": data['seriess'][0]['units'],
                "observation_start": data['seriess'][0]['observation_start'],
                "last_update_local_time": fred_local_time.strftime("%Y-%m-%d %H:%M:%S"),
                "last_update_utc":fred_utc_time.strftime("%Y-%m-%d %H:%M:%S"),
            }

            results.append(fred_series_data)

    return results


requester_fred_series()

## Get data for db table `FREDObservations`

- SeriesID
- ObservationDate
- NativeValue
- DateFetched `TEXT NOT NULL DEFAULT (datetime('now')),`

<s><b style="color:#fa0240;">Use for `observation_start` date the earliest common date among all data series</b></s>

<b style="color:DodgerBlue;">Just redownload everything every month since it is not so much</b>

In [ ]:
earliest_common_observation_start = "2000-12-01"

def requester_fred_observations():
    url = 'https://api.stlouisfed.org/fred/series/observations'
    results = []

    with requests.Session() as session:
        for series_id in SERIES_IDS:
            params = {
                "api_key": API_KEY,
                "file_type": "json",
                "sort_order": "asc",
                "units": "lin",
                "observation_start": earliest_common_observation_start,
                "series_id": series_id
            }

            r = session.get(url=url, params=params, timeout=30)
            r.raise_for_status()

            data = r.json()
            entries = data['observations']

            for entry in entries:
                fred_observations_data = {
                    "series_id": series_id,
                    "observation_date": entry['date'],
                    "native_value": entry['value']
                }
            
                results.append(fred_observations_data)

    return results

## Prepare data from `FREDObservations` for db table `MonthlyFeatures`
- SeriesID
- MonthYear
- MonthlyValue
- DateCalculated `TEXT NOT NULL DEFAULT (datetime('now')),`

<b style="color:#fa0240;">Implement a checker if the dataset for the month is complete to kick of the ML pipeline</b>

In [ ]:
native_observations = database.db_requester_fetchall("all_obs_FREDObservations.sql")

In [ ]:
# Convert all dates to YYYY-MM and timestamp is dropped
arr = np.array(native_observations)
arr = np.delete(arr,[False, False, False, True], 1)
arr[:][:,1] = arr[:][:,1].astype("datetime64[M]")
# arr
idx_icsa = np.where(arr=='ICSA')[0]
none_icsa_arr = np.delete(arr, idx_icsa, 0)
none_icsa_arr

In [ ]:
icsa_arr = arr[idx_icsa]

df_icsa = pd.DataFrame(icsa_arr)
column_names = ['series_id', 'month_year', 'monthly_value']
df_icsa.columns = column_names

df_monthly_icsa = (
    df_icsa.groupby("month_year", as_index=False).agg({
        "series_id": "last",
        "monthly_value": lambda x: None if x.isna().any() else x.sum()
    })
)
df_monthly_icsa.insert(
    0,
    df_monthly_icsa.columns[1],
    df_monthly_icsa.pop(df_monthly_icsa.columns[1])
)

df_monthly_icsa

In [ ]:
df_none_ics = pd.DataFrame(none_icsa_arr)
df_none_ics.columns = column_names
df_none_ics

In [ ]:
df_all_monthly = pd.concat([df_none_ics, df_monthly_icsa],ignore_index=True)
df_all_monthly = df_all_monthly.astype(object).where(
    pd.notna(df_all_monthly),
    None
)
db_ready_monthly_values = df_all_monthly.to_dict(orient='records')
df_all_monthly

In [ ]:
df_all_monthly[df_all_monthly['monthly_value'].isnull()]

In [ ]:
db_ready_monthly_values[0:2]

# Old

In [ ]:
base_url = 'https://api.stlouisfed.org/fred/series/observations'

base_params = {
    'api_key': API_KEY,
    'file_type': 'json',
    'sort_order': 'asc',
    "units": "lin"
}

def requester(series_id, observation_start, observation_end='9999-12-31', url=base_url):
    params = base_params.copy()
    params.update({
        'series_id': series_id,
        'observation_start': observation_start,
        'observation_end': observation_end
    })

    r = requests.get(url=url, params=params)

    if r.status_code == 200:
        return r.json()
    else:
        status = f"{params['series_id']} raised an error code {r.status_code}"
        return status

In [ ]:
base_params

In [ ]:
response = requester('ICSA', '2000-01-01')

In [ ]:
response.keys()

In [ ]:
df = pd.DataFrame(response['observations'])
df

# Testing

In [ ]:
payload = {
    'series_id': 'ICSA',
    'api_key': API_KEY,
    'file_type': 'json'
}

r = requests.get('https://api.stlouisfed.org/fred/series', params=payload)

In [ ]:
print(r.json())
print(f'observations_start: {r.json()['seriess'][0]['id']}')
print(f'observations_start: {r.json()['seriess'][0]['observation_start']}')
print(f'observations_end: {r.json()['seriess'][0]['observation_end']}')
print(f'frequency: {r.json()['seriess'][0]['frequency']}')
print(f'frequency_short: {r.json()['seriess'][0]['frequency_short']}')
print(f'units: {r.json()['seriess'][0]['units']}')



In [ ]:
r.json()

In [ ]:
payload.update({'frequency': 'm'})
payload.update({'sort_order': 'desc'})
r_obsv = requests.get(base_url, params=payload)

In [ ]:
print(r_obsv.status_code)

In [ ]:
arr = np.array(native_observations)
icsa_arr = arr[np.where(arr=='ICSA')[0]]

dates = icsa_arr[:][:,1].astype("datetime64[M]")

icsa_arr[:][:,1] = dates

unique_months = np.unique(dates)
values = icsa_arr[:,2]

monthly_icsa_arr = np.array([
    values[dates == month].sum()
    for month in unique_months
])

monthly_icsa_arr

# dates = arr[idx[0]][:18][:,1].astype("datetime64[D]")

